## Import Datas

In [ ]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/Thèse de doctorat/Redaction - rapports et articles/Articles de conférence redigés/ICATH 2026/code/data/preprocessed options datas/"

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
FinanceUnderlying = [ "BAC", "C", "GS", "JPM", "WFC" ]
HealthUnderlying = [ "ABBV", "JNJ", "MRK", "PFE", "UNH" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + FinanceUnderlying + HealthUnderlying + TechnologyUnderlying

In [ ]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "BAC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "C" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GS" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JPM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "WFC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "ABBV" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JNJ" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MRK" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "PFE" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "UNH" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [ ]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [ ]:
import pandas as pd
from scipy.stats import norm
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return price

def compute_bs_price(ticker, option_type):
  data = dataset[ticker]["test"]
  data = data[data["type"] == option_type]
  test_data = data[['time-to-maturity', 'type', 'riskFreeRate', 'sousJacent', 'impliedVolatility', 'strike', "lastPrice"]]

  test_data['BS_price'] = test_data.apply(
    lambda row: black_scholes(
        S=row['sousJacent'],
        K=row['strike'],
        T=row['time-to-maturity'],
        r=row['riskFreeRate'],
        sigma=row['impliedVolatility'],
        option_type=row['type']
    ),
    axis=1
)
  return test_data



In [ ]:
def compute_bs_price(ticker, option_type):
  data = dataset[ticker]["test"]
  test_data = data[data["optionType"] == option_type]

  test_data['BS_price'] = test_data.apply(
    lambda row: black_scholes(
        S=row['underlyingLastPrice'],
        K=row['strike'],
        T=row['timeToMaturity'],
        r=row['riskFreeRate'],
        sigma=row['impliedVolatility'],
        option_type=row['optionType']
    ),
    axis=1
  )

  rmse = np.sqrt(mean_squared_error(test_data['lastPrice'], test_data['BS_price']))
  print(f"ticker: {ticker}, option type: {option_type}, RMSE: {rmse:.3f}")

## option price using black & scholes model

In [ ]:
ticker = "MSFT"
option_type="put"

In [ ]:
compute_bs_price(ticker, option_type)

ticker: MSFT, option type: put, RMSE: 20.226


/tmp/ipykernel_1006/1149508461.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data['BS_price'] = test_data.apply(
